In [0]:
from dataclasses import dataclass
from pyspark.sql import DataFrame
from delta.tables import DeltaTable

def idempotencia(df: DataFrame, tabela: str):
    try:
        if spark.catalog.tableExists(tabela):

            delta_table = DeltaTable.forName(spark, tabela)

            (
                delta_table.alias("destino")
                .merge(
                    df.alias("origem"),
                    "destino.id = origem.id"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )

        else:
            (
                df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(tabela)
            )

        return True

    except Exception as e:
        print(f"Error: {e}")
        return False

def salvar_delta(df_cliente: DataFrame, df_contas: DataFrame, df_emprestimos: DataFrame, df_pagamentos: DataFrame, df_transferencias: DataFrame):
    try:
        #salvar clientes
        tabela_cliente = "dbw_banco_orion.bronze.clientes"
        if not idempotencia(df_cliente ,tabela_cliente):
            raise Exception("Error ao salvar os dados na delta")

        #savar contas
        tabela_contas = "dbw_banco_orion.bronze.contas"
        if not idempotencia(df_contas, tabela_contas):
            raise Exception("Error ao salvar os dados na delta")

        #Salvar emprestimos
        tabela_emprestimos ="dbw_banco_orion.bronze.emprestimos"
        if not idempotencia(df_emprestimos, tabela_emprestimos):
            raise Exception("Error ao salvar os dados na delta")

        #Salvar pagamentos
        tabela_pagamentos = "dbw_banco_orion.bronze.pagamentos"
        if not idempotencia(df_pagamentos, tabela_pagamentos):
            raise Exception("Error ao salvar os dados na delta")

        #Salvar transferencias
        tabela_transferencias = "dbw_banco_orion.bronze.transferencias"
        if not idempotencia(df_transferencias, tabela_transferencias):
            raise Exception("Error ao salvar os dados na delta")

        print("Dados salvos com sucesso!")
        return True
    except Exception as e:
        print(f"Error: {e}")
        return False

df_cliente = spark.read.parquet("/Volumes/dbw_banco_orion/bronze/dados_brutos/clientes.parquet")
df_contas = spark.read.parquet("/Volumes/dbw_banco_orion/bronze/dados_brutos/contas.parquet")
df_emprestimos = spark.read.parquet("/Volumes/dbw_banco_orion/bronze/dados_brutos/emprestimos.parquet")
df_pagamentos = spark.read.parquet("/Volumes/dbw_banco_orion/bronze/dados_brutos/pagamentos.parquet")
df_transferencias = spark.read.parquet("/Volumes/dbw_banco_orion/bronze/dados_brutos/transferencias.parquet")
if not salvar_delta(df_cliente, df_contas, df_emprestimos, df_pagamentos, df_transferencias):
    raise Exception("Error ao salvar os dados na delta")
